# S5.5 Citation Enforcement — Interactive Verification

This notebook verifies the citation enforcement module (`src/services/rag/citation.py`).

**What it does:**
- Parses inline `[N]` citations from LLM output
- Validates citations against the source list
- Formats a standardized markdown source list with arXiv links
- Strips LLM-generated "Sources:" sections and replaces with standardized format
- Supports both single-shot and streaming modes

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.rag.citation import (
    CitationResult,
    CitationValidation,
    enforce_citations,
    format_source_list,
    parse_citations,
    stream_with_citations,
    validate_citations,
)
from src.services.rag.models import LLMMetadata, RAGResponse, RetrievalMetadata, SourceReference

print("\u2713 All imports successful")

✓ All imports successful


## FR-1: Citation Parser

In [2]:
# Test basic citation extraction
text = "Transformers [1] revolutionized NLP. BERT [2] and GPT [3] are notable examples."
result = parse_citations(text)
assert result == [1, 2, 3], f"Expected [1,2,3], got {result}"
print(f"Parsed citations: {result}")

# Duplicates are deduplicated
text2 = "Attention [1] is key [1] in transformers [2]."
result2 = parse_citations(text2)
assert result2 == [1, 2], f"Expected [1,2], got {result2}"
print(f"Deduplicated: {result2}")

# Invalid indices ignored
text3 = "Invalid [0] and [-1] and [abc]. Valid [1] and [5]."
result3 = parse_citations(text3)
assert result3 == [1, 5], f"Expected [1,5], got {result3}"
print(f"Invalid filtered: {result3}")

# Nested brackets [[1]] still extracted
text4 = "Nested [[1]] brackets [[2]]."
result4 = parse_citations(text4)
assert result4 == [1, 2], f"Expected [1,2], got {result4}"
print(f"Nested brackets: {result4}")

# Range-style [1-3] ignored
text5 = "See [1-3] for details. Also [2]."
result5 = parse_citations(text5)
assert result5 == [2], f"Expected [2], got {result5}"
print(f"Range ignored: {result5}")

print("\n\u2713 FR-1 Citation Parser: all checks passed")

Parsed citations: [1, 2, 3]
Deduplicated: [1, 2]
Invalid filtered: [1, 5]
Nested brackets: [1, 2]
Range ignored: [2]

✓ FR-1 Citation Parser: all checks passed


## FR-2: Citation Validator

In [3]:
# Helper to create sources
def make_source(index, arxiv_id=None, title=None, authors=None):
    return SourceReference(
        index=index,
        arxiv_id=arxiv_id or f"230{index}.0000{index}",
        title=title or f"Paper {index}",
        authors=authors or [f"Author{index}A", f"Author{index}B"],
        arxiv_url=f"https://arxiv.org/abs/{arxiv_id or f'230{index}.0000{index}'}",
        chunk_text=f"Chunk {index}",
        score=0.9,
    )

sources = [make_source(1), make_source(2), make_source(3)]

# All valid
v = validate_citations([1, 2, 3], sources)
assert v.is_valid is True
assert v.citation_coverage == 1.0
assert v.invalid_citations == []
print(f"All valid: coverage={v.citation_coverage}, is_valid={v.is_valid}")

# Hallucinated citation
v2 = validate_citations([1, 5], sources)
assert v2.is_valid is False
assert v2.invalid_citations == [5]
assert v2.uncited_sources == [2, 3]
print(f"Hallucinated: valid={v2.valid_citations}, invalid={v2.invalid_citations}")

# No citations, no sources = valid
v3 = validate_citations([], [])
assert v3.is_valid is True
print(f"Both empty: is_valid={v3.is_valid}")

# No citations but sources exist = invalid
v4 = validate_citations([], sources)
assert v4.is_valid is False
assert v4.uncited_sources == [1, 2, 3]
print(f"No cites with sources: is_valid={v4.is_valid}, uncited={v4.uncited_sources}")

print("\n\u2713 FR-2 Citation Validator: all checks passed")

All valid: coverage=1.0, is_valid=True
Hallucinated: valid=[1], invalid=[5]
Both empty: is_valid=True
No cites with sources: is_valid=False, uncited=[1, 2, 3]

✓ FR-2 Citation Validator: all checks passed


## FR-3: Source List Formatter

In [4]:
# Basic formatting
sources = [
    make_source(1, arxiv_id="2301.00001", title="Attention Is All You Need", authors=["Vaswani", "Shazeer", "Parmar"]),
    make_source(2, arxiv_id="1810.04805", title="BERT", authors=["Devlin", "Chang", "Lee", "Toutanova"]),
]
result = format_source_list(sources)
print("=== Basic Format ===")
print(result)
assert "**Sources:**" in result
assert "[Attention Is All You Need]" in result
assert "Vaswani, Shazeer, Parmar" in result  # <= 3 authors: all listed
assert "Devlin et al." in result  # > 3 authors: et al.
assert "2023" in result  # year from 2301
assert "2018" in result  # year from 1810

# cited_only filter
result2 = format_source_list(sources, cited_indices=[1])
print("\n=== Cited Only (index 1) ===")
print(result2)
assert "Attention Is All You Need" in result2
assert "BERT" not in result2

# Empty sources
assert format_source_list([]) == ""

# Missing title falls back to arxiv_id
s = [make_source(1, arxiv_id="2301.99999", title="")]
result3 = format_source_list(s)
print("\n=== Missing Title ===")
print(result3)
assert "2301.99999" in result3

print("\n\u2713 FR-3 Source List Formatter: all checks passed")

=== Basic Format ===
**Sources:**
1. [Attention Is All You Need](https://arxiv.org/abs/2301.00001) — Vaswani, Shazeer, Parmar, 2023
2. [BERT](https://arxiv.org/abs/1810.04805) — Devlin et al., 2018

=== Cited Only (index 1) ===
**Sources:**
1. [Attention Is All You Need](https://arxiv.org/abs/2301.00001) — Vaswani, Shazeer, Parmar, 2023

=== Missing Title ===
**Sources:**
1. [Paper 1](https://arxiv.org/abs/2301.99999) — Author1A, Author1B, 2023

✓ FR-3 Source List Formatter: all checks passed


## FR-4: Citation Enforcer (Full Pipeline)

In [5]:
# Full enforcement flow
sources = [
    make_source(1, arxiv_id="1706.03762", title="Attention Is All You Need", authors=["Vaswani", "Shazeer"]),
    make_source(2, arxiv_id="1810.04805", title="BERT", authors=["Devlin", "Chang", "Lee", "Toutanova"]),
    make_source(3, arxiv_id="2005.14165", title="GPT-3", authors=["Brown", "Mann", "Ryder"]),
]

answer = """Transformers [1] introduced the self-attention mechanism that replaced recurrence.
BERT [2] applied bidirectional pre-training, while GPT-3 [3] scaled to 175B parameters.

**Sources:**
1. Some LLM-generated source info
2. Another LLM source
3. Yet another"""

response = RAGResponse(
    answer=answer,
    sources=sources,
    query="What are transformers?",
    retrieval_metadata=RetrievalMetadata(),
    llm_metadata=LLMMetadata(),
)

result = enforce_citations(response)

print("=== Formatted Answer ===")
print(result.formatted_answer)
print("\n=== Validation ===")
print(f"  is_valid: {result.validation.is_valid}")
print(f"  coverage: {result.validation.citation_coverage}")
print(f"  valid: {result.validation.valid_citations}")
print(f"  invalid: {result.validation.invalid_citations}")
print(f"  uncited: {result.validation.uncited_sources}")

# Assertions
assert result.formatted_answer.count("**Sources:**") == 1, "Should have exactly one Sources section"
assert result.validation.is_valid is True
assert result.validation.citation_coverage == 1.0
assert "Attention Is All You Need" in result.formatted_answer
assert "arxiv.org/abs/1706.03762" in result.formatted_answer
assert "Devlin et al." in result.formatted_answer  # >3 authors
assert "Some LLM-generated" not in result.formatted_answer, "LLM sources should be stripped"

print("\n\u2713 FR-4 Citation Enforcer: all checks passed")

=== Formatted Answer ===
Transformers [1] introduced the self-attention mechanism that replaced recurrence.
BERT [2] applied bidirectional pre-training, while GPT-3 [3] scaled to 175B parameters.

**Sources:**
1. [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — Vaswani, Shazeer, 2017
2. [BERT](https://arxiv.org/abs/1810.04805) — Devlin et al., 2018
3. [GPT-3](https://arxiv.org/abs/2005.14165) — Brown, Mann, Ryder, 2020

=== Validation ===
  is_valid: True
  coverage: 1.0
  valid: [1, 2, 3]
  invalid: []
  uncited: []

✓ FR-4 Citation Enforcer: all checks passed


## FR-5: Streaming Citation Support

In [6]:
sources = [
    make_source(1, arxiv_id="1706.03762", title="Attention Is All You Need", authors=["Vaswani", "Shazeer"]),
    make_source(2, arxiv_id="1810.04805", title="BERT", authors=["Devlin", "Chang"]),
]

async def mock_llm_stream():
    yield "Transformers [1] use self-attention. "
    yield "BERT [2] applies bidirectional training.\n\n"
    yield "**Sources:**\n1. LLM source\n"

streamer = stream_with_citations(mock_llm_stream(), sources)
tokens = []
async for token in streamer:
    tokens.append(token)

full = "".join(tokens)
print("=== Streamed Output ===")
print(full)

assert full.count("**Sources:**") == 1, "Exactly one Sources section"
assert "LLM source" not in full, "LLM sources stripped"
assert "Attention Is All You Need" in full
assert "arxiv.org/abs/1706.03762" in full
assert streamer.validation is not None
print(f"\nValidation: is_valid={streamer.validation.is_valid}, coverage={streamer.validation.citation_coverage}")

print("\n\u2713 FR-5 Streaming Citation Support: all checks passed")

=== Streamed Output ===
Transformers [1] use self-attention. BERT [2] applies bidirectional training.

**Sources:**
1. [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — Vaswani, Shazeer, 2017
2. [BERT](https://arxiv.org/abs/1810.04805) — Devlin, Chang, 2018

Validation: is_valid=True, coverage=1.0

✓ FR-5 Streaming Citation Support: all checks passed
